# Yancc vs SFINCS Comparison - Part 6: VMEC Convergence and Radial Ambipolar Scan

This notebook includes AI generated code (Claude)

This notebook combines convergence scanning and radial ambipolar Er scanning using a
VMEC equilibrium (W7-X) instead of a DESC equilibrium. It builds on:
- **Part 4**: Radial ambipolar scan with I/O (DESC equilibrium, SFINCS profiles)
- **Part 5**: Convergence scan demonstration (DESC equilibrium, simple profiles)
- **Yancc test case VMEC**: Direct VMEC equilibrium usage

The workflow is organized to first run convergence scans (to validate resolution
settings), then run the full radial ambipolar scan with confidence in the numerics.

---
## 1. Imports and Setup

In [ ]:
import numpy as np
import logging
from pathlib import Path

import yancc
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import GlobalMaxwellian
from yancc.yancctools import (
    sfincstools,
    yancctools,
    yancctools_plot,
    yancctools_io,
    run_convergence_scan,
    plot_convergence_scan,
    save_results,
    load_results,
)

# Configure logging to see scan progress
logging.basicConfig(level=logging.INFO, force=True)

---
## 2. Configuration

All configuration is collected into an `options` dictionary (JSON/YAML compatible).
The VMEC equilibrium path replaces the DESC equilibrium used in earlier notebooks.

In [ ]:
# VMEC equilibrium
vmec_path = "/u/npablant/data/w7x/vmec/w7x_ref_172/wout.nc"
eq_type = "vmec"

# SFINCS profiles
profile_file = "/u/npablant/analysis/w7x/171207006/stelltran/run07/sfincs/t1.1273/profiles"

# Shared options for both convergence and ambipolar scans
options = {
    "output_path": "/u/npablant/analysis/yancc/runs/",
    "nx": 11,               # Speed resolution
    "na": 91,               # Pitch-angle resolution
    "nt": 17,               # Poloidal resolution
    "nz": 33,               # Toroidal resolution
    "erho_min": -15000.0,   # Coarse Er scan range [V/m]
    "erho_max": 15000.0,
    "erho_num": 41,         # Points in coarse Er scan
    "convergence_scan_steps": 5,
}

---
## 3. Load and Scale Kinetic Profiles

Load polynomial profile coefficients from the SFINCS `profiles` file and scale to
SI units (SFINCS profiles are in 10^20 m^-3 and keV).

In [ ]:
# Generate the raw polynomial functions (dimensionless coefficients)
funcs = sfincstools.generate_profile_functions(profile_file)

# Define scaled functions for yancc
# Species 1: Electrons, Species 2: Hydrogen
density_e = lambda r: funcs["species_1_density"](r) * 1e20
temp_e = lambda r: funcs["species_1_temperature"](r) * 1e3

density_H = lambda r: funcs["species_2_density"](r) * 1e20
temp_H = lambda r: funcs["species_2_temperature"](r) * 1e3

# Define global species objects
global_species = [
    GlobalMaxwellian(yancc.species.Electron, temperature=temp_e, density=density_e),
    GlobalMaxwellian(yancc.species.Hydrogen, temperature=temp_H, density=density_H),
]

print("Profiles loaded and species defined.")

In [ ]:
# Visualize the loaded profiles
sfincstools.plot_profiles_plotly(funcs)

---
## 4. Convergence Scan

Run a convergence scan at a single radial surface to validate that the baseline
resolution settings (`nx`, `na`, `nt`, `nz`) are adequate before committing to the
full radial scan.

- **rho = 0.2** (inner radial surface)
- **Er = 0.0 V/m** (zero radial electric field)

In [ ]:
convergence_results = run_convergence_scan(
    eq_type=eq_type,
    eq_data=vmec_path,
    global_species=global_species,
    rho=0.2,
    erho=0.0,
    options=options,
)

convergence_runid = convergence_results["runid"]
print(f"Convergence scan completed. RunID: {convergence_runid}")

In [ ]:
# Save convergence results
convergence_run_dir = Path(options["output_path"]) / convergence_runid
yancctools_io.save_options(options, convergence_run_dir)
save_results(convergence_results, convergence_run_dir)
print(f"Convergence results saved to: {convergence_run_dir}")

In [ ]:
# Plot convergence results
fig = plot_convergence_scan(convergence_results)

In [ ]:
# Validation: reload and re-plot to verify I/O round-trip
convergence_loaded = load_results(convergence_run_dir / "results.h5")
fig = plot_convergence_scan(convergence_loaded)

---
## 5. Radial Ambipolar Scan

Using the resolution settings validated by the convergence scan above, perform a
radial scan over multiple surfaces to find the ambipolar radial electric field
profile.

In [ ]:
# Define radial grid and velocity grids
rho_grid = np.linspace(0.2, 0.8, 5)

speedgrid = MaxwellSpeedGrid(nx=options["nx"])
pitchgrid = UniformPitchAngleGrid(nxi=options["na"])

In [ ]:
# Execute radial ambipolar scan
ambipolar_results = yancctools.scan_ambipolar_profile(
    rho_grid,
    eq_type=eq_type,
    eq_data=vmec_path,
    pitchgrid=pitchgrid,
    speedgrid=speedgrid,
    global_species=global_species,
    options=options,
)

ambipolar_runid = ambipolar_results["runid"]
print(f"Radial scan completed. RunID: {ambipolar_runid}")

In [ ]:
# Save options, inputs, and results
ambipolar_run_dir = Path(options["output_path"]) / ambipolar_runid
print(f"Saving run data to: {ambipolar_run_dir}")

# 1. Save Options (JSON and YAML)
yancctools_io.save_options(options, ambipolar_run_dir)

# 2. Save Inputs (HDF5)
inputs = {
    "rho_grid": rho_grid,
    "profiles": {
        "rho": rho_grid,
        "Te": [temp_e(r) for r in rho_grid],
        "ne": [density_e(r) for r in rho_grid],
        "Ti": [temp_H(r) for r in rho_grid],
        "ni": [density_H(r) for r in rho_grid],
    },
    "grids": {
        "pitch_nxi": pitchgrid.nxi,
        "speed_nx": speedgrid.nx,
    },
    "equilibrium": f"VMEC: {vmec_path}",
    "source_profile_file": profile_file,
}
yancctools_io.save_inputs(inputs, ambipolar_run_dir)

# 3. Save Results (HDF5)
yancctools_io.save_results(ambipolar_results, ambipolar_run_dir)

In [ ]:
# Validation: reload from disk and plot
loaded_results = yancctools_io.load_results(ambipolar_run_dir / "results.h5")
print("Results reloaded from disk.")

yancctools_plot.plot_ambipolar_summary(loaded_results, global_species)